# CloudGuardian: AI-Powered Predictive Maintenance and Anomaly Detection
## College Milestone 1 Demonstration: Ingestion, Drain3 Parsing & 3D Tensor Assembly

**Institution:** Ajay Kumar Garg Engineering College (AKGEC, Ghaziabad)  
**Affiliation:** Dr. A.P.J. Abdul Kalam Technical University (AKTU, Lucknow)  
**Supervisor:** Dr. Shashank Sahu  
**Group 10 Team Members:**
- Ayush Kumar Agrawal (2300270100043)
- Anubhav Hajela (2300270100025)
- Arpit Kumar (2300270100033)
- Amit (2300270100015)

---
### Milestone 1 Objectives:
1. Live stream ingestion and regex redaction of dynamic/sensitive tokens (`<IP>`, `<NUM>`, `<PATH>`, `<HEX>`, `<BLOCK_ID>`).
2. Dynamic template mining using a fixed-depth parse tree (**Drain3**) to extract the canonical HDFS templates ($E_1$ to $E_{29}$).
3. Synchronizing continuous Prometheus metrics and discrete template frequencies onto a uniform 30-second grid.
4. Constructing and validating the 3D PyTorch tensor of shape **`torch.Size([32, 60, 42])`** ($N=32$ samples, $T=60$ ticks = 30 mins, $F=42$ features).

In [ ]:
# 1. Setup Python Path and Import Modules
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
try:
    import torch
    print(f"PyTorch Version: {torch.__version__}")
except ImportError:
    import torch

from cloudguardian.ingestion.redactor import LogRedactor
from cloudguardian.ingestion.drain_parser import DrainParser
from cloudguardian.ingestion.synchronizer import MetricLogSynchronizer
from cloudguardian.features.window_buffer import SlidingWindowBuffer

print("CloudGuardian Phase 1 Modules Loaded Successfully!")

### Step 1: Regex Normalization and Token Redaction
Raw logs contain variable information like dynamic IPs, memory pointers, and port numbers that cause template explosion. `LogRedactor` standardizes these into tokens while preserving structural syntax.

In [ ]:
redactor = LogRedactor()
sample_log = "081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687 src: 10.251.43.115:50010 dest: 10.251.43.115:50010"
redacted_log, meta = redactor.process(sample_log)

print("RAW LOG:")
print(sample_log)
print("\nREDACTED LOG:")
print(redacted_log)
print("\nEXTRACTED METADATA:")
print(meta)

### Step 2: Extracting Canonical Templates ($E_1$ to $E_{29}$) via Drain3 Tree
We process 29 canonical log events from the Hadoop Distributed File System (HDFS) benchmark dataset.

In [ ]:
parser = DrainParser(depth=4, similarity_threshold=0.5)

hdfs_logs = [
    "081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687 src: 10.251.43.115:50010 dest: 10.251.43.115:50010",
    "081109 203519 144 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687 terminating",
    "081109 203520 145 INFO dfs.DataNode$DataXceiver: Received block blk_-1608999687 of size 67108864 from 10.251.43.115:50010",
    "081109 203521 146 INFO dfs.DataNode$DataXceiver: Received block blk_-1608999687 src: 10.251.43.115:50010 dest: 10.251.43.115:50010 of size 67108864",
    "081109 203522 147 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_0001/part-00001. blk_358750814",
    "081109 203523 148 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_358750814 terminating",
    "081109 203524 149 INFO dfs.DataBlockScanner: Verification succeeded for block blk_358750814",
    "081109 203525 150 INFO dfs.FSNamesystem: BLOCK* ask 10.251.43.115:50010 to replicate blk_358750814 to datanode(s) 10.251.203.14:50010",
    "081109 203526 151 INFO dfs.DataNode$DataTransfer: Starting thread to transfer block blk_358750814 to 10.251.203.14:50010",
    "081109 203527 152 INFO dfs.DataNode$DataXceiver: Deleting block blk_358750814 file /mnt/hadoop/dfs/data/current/blk_358750814",
    "081109 203528 153 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.43.115:50010 is added to blk_358750814 size 67108864",
    "081109 203529 154 INFO dfs.DataNode$DataXceiver: Served block blk_358750814 to 10.251.43.115:50010",
    "081109 203530 155 INFO dfs.FSNamesystem: BLOCK* ask 10.251.43.115:50010 to delete blk_358750814",
    "081109 203531 156 WARN dfs.DataNode$DataXceiver: writeBlock blk_358750814 received exception java.io.IOException: Broken pipe",
    "081109 203532 157 ERROR dfs.DataNode$DataXceiver: Exception in receiveBlock for block blk_358750814 java.io.EOFException",
    "081109 203533 158 INFO dfs.DataNode$FSDataset: Changing block file offset of block blk_358750814 from 0 to 4096",
    "081109 203534 159 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: Redundant addStoredBlock request received for blk_358750814 on 10.251.43.115:50010",
    "081109 203535 160 INFO dfs.FSNamesystem: BLOCK* ask 10.251.43.115:50010 to replicate blk_358750814 to datanode 10.251.43.201:50010",
    "081109 203536 161 INFO dfs.DataNode$FSDataset: Reopen Block blk_358750814",
    "081109 203537 162 WARN dfs.DataNode$FSDataset: Unexpected error trying to delete block blk_358750814. BlockInfo not found in volumeMap.",
    "081109 203538 163 WARN dfs.FSNamesystem: PendingReplicationMonitor timed out block blk_358750814",
    "081109 203539 164 INFO dfs.DataNode$FSDataset: Moved block blk_358750814 to /mnt/hadoop/dfs/data/current/subdir12",
    "081109 203540 165 ERROR dfs.DataNode$DataTransfer: Failed to transfer blk_358750814 to 10.251.203.14:50010 got java.net.SocketTimeoutException",
    "081109 203541 166 INFO dfs.DataNode: Waiting for threadpool to drain before shutting down datanode 10.251.43.115:50010",
    "081109 203542 167 INFO dfs.DataNode: Block pool BP-19284128-10.251.43.115-14298129 terminating",
    "081109 203543 168 WARN dfs.FSNamesystem: Heartbeat monitor reported dead node 10.251.43.115:50010",
    "081109 203544 169 FATAL dfs.DataNode: Disk volume failure on directory /mnt/hadoop/dfs/data",
    "081109 203545 170 INFO dfs.FSNamesystem: SafeModeMonitor: STAGE 1: Leaving safe mode with 12054 blocks",
    "081109 203546 171 INFO dfs.NetworkTopology: Network topology update: node 10.251.43.115:50010 switched rack to /rack-default"
]

print(f"Extracted {len(hdfs_logs)} Log Templates:\n")
for line in hdfs_logs:
    red, _ = redactor.process(line)
    res = parser.parse(red)
    print(f"{res['template_id']:<5} -> {res['template_str']}")

print(f"\nTotal Canonical Templates Mined: {parser.total_templates()}")

### Step 3: Multi-Tick Synchronization & Window Assembly
Synchronizes Prometheus metrics and log event occurrences into 30-second ticks, building 60-tick sliding windows (30 minutes) into a batch of 32.

In [ ]:
NUM_TEMPLATES = 29
TICK_SEC = 30.0
WINDOW_SIZE = 60
BATCH_SIZE = 32
TOTAL_TICKS = BATCH_SIZE + WINDOW_SIZE - 1  # 91 ticks

sync = MetricLogSynchronizer(tick_duration_sec=TICK_SEC, max_templates=NUM_TEMPLATES, start_time=1700000000.0)
window_buf = SlidingWindowBuffer(window_size=WINDOW_SIZE, step_size=1)

for t in range(TOTAL_TICKS):
    ts = 1700000000.0 + (t * TICK_SEC)
    
    # Sample metric telemetry
    metrics = {
        "cpu_util": 35.0 + 10.0 * np.sin(t / 8.0),
        "mem_util": 45.0 + (t * 0.4),
        "disk_latency_ms": 5.0,
        "net_throughput_mb": 20.0,
        "conn_wait_ms": 2.5 + (15.0 if t > 70 else 0.0)
    }
    sync.ingest_metric(ts, metrics)
    
    # Sporadic log templates (E1..E29)
    for _ in range(np.random.randint(1, 5)):
        t_num = np.random.randint(1, 30)
        lvl = "ERROR" if t_num in [14, 15, 23, 27] else "INFO"
        sync.ingest_log(ts + np.random.uniform(1.0, 25.0), template_num=t_num, level=lvl)
        
    features = sync.get_tick_features(t)
    # Append session rate feature to reach 42 channels
    features.append(features[-3] / TICK_SEC)
    window_buf.append_tick(features, timestamp=ts)

windows_np, _ = window_buf.generate_all_windows()
print(f"Generated Windows Array: {windows_np.shape}")

### Step 4: PyTorch Tensor Shape & Semantic Validation
We convert the batch array into a PyTorch tensor and verify the exact shape: `torch.Size([32, 60, 42])`.

In [ ]:
import torch
tensor_3d = torch.from_numpy(windows_np)

print("=" * 60)
print("  PYTORCH 3D TENSOR VERIFICATION")
print("=" * 60)
print(f"Tensor Type:             {type(tensor_3d)}")
print(f"Validated Tensor Size:   {tensor_3d.shape}")
print(f"Data Type:               {tensor_3d.dtype}")
print(f"Missing / NaN Count:     {int(torch.isnan(tensor_3d).sum())}")
print(f"Inf Value Count:         {int(torch.isinf(tensor_3d).sum())}")

# Assertions
assert tensor_3d.shape == torch.Size([32, 60, 42]), f"Expected [32, 60, 42], got {tensor_3d.shape}"
assert tensor_3d.dtype == torch.float32
print("\nSUCCESS: Validated 3D PyTorch Tensor shape is EXACTLY torch.Size([32, 60, 42])!")

### Step 5: Feature Channel Semantics
- **Channels 0 – 4 (5 features):** Mean system metrics (CPU %, Mem %, Disk Latency, Net MB, Conn Wait Latency)
- **Channels 5 – 33 (29 features):** Canonical HDFS template counts ($E_1$ through $E_{29}$)
- **Channels 34 – 37 (4 features):** Inter-event time deltas within tick (Mean, Std, Min, Max)
- **Channels 38 – 41 (4 features):** Metadata (Total logs, Unique templates, Error ratio, Session rate)
**Total:** 42 Features.